# Stage 6 - Explainability and Output Report
This notebook computes SHAP explanations, builds per-applicant report generation, and prints natural-language summaries.

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PLOT_DIR = BASE_DIR / 'outputs' / 'plots' / 'stage6'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def save_plot(name: str):
    path = PLOT_DIR / f'{name}.png'
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Saved plot: {path}')


## Load Stage 5 Outputs

In [ ]:
with open(PROCESSED_DIR / 'stage5_outputs.pkl', 'rb') as f:
    stage5 = pickle.load(f)
best_model = stage5['best_model']
best_model_name = stage5['best_model_name']
X_train_eng = stage5['X_train_eng']
X_test_eng = stage5['X_test_eng']
y_test = stage5['y_test']
cluster_model = stage5['cluster_model']
cluster_names = stage5['cluster_names']
borrower_type_test = stage5['borrower_type_test']
print('Best model:', best_model_name)
print('X_test shape:', X_test_eng.shape)


## Step 1 - Compute SHAP Values
Use model-appropriate explainer and compute SHAP values on the full test set.

In [ ]:
if best_model_name in ['random_forest', 'xgboost']:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test_eng)
elif best_model_name in ['lr_l2', 'lr_l1']:
    explainer = shap.LinearExplainer(best_model, X_train_eng)
    shap_values = explainer.shap_values(X_test_eng)
elif best_model_name == 'svm':
    background = shap.sample(X_train_eng, 200, random_state=42)
    explainer = shap.KernelExplainer(lambda x: best_model.predict_proba(pd.DataFrame(x, columns=X_test_eng.columns))[:, 1], background)
    shap_values = explainer.shap_values(X_test_eng)
else:
    explainer = shap.Explainer(best_model, X_train_eng)
    shap_values = explainer(X_test_eng).values

if isinstance(shap_values, list):
    shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
print('SHAP values shape:', np.array(shap_values).shape)


## Step 2 - Global SHAP Plots
Generate beeswarm, bar summary, and dependence plots for top 3 global features.

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_eng, show=False)
plt.title('Stage 6 SHAP Beeswarm Summary')
save_plot('stage6_shap_beeswarm')
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_eng, plot_type='bar', show=False)
plt.title('Stage 6 SHAP Bar Summary')
save_plot('stage6_shap_bar')
plt.show()

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[-3:][::-1]
top3_features = X_test_eng.columns[top3_idx].tolist()
print('Top 3 SHAP features:', top3_features)

for feat in top3_features:
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(feat, shap_values, X_test_eng, show=False)
    plt.title(f'Stage 6 SHAP Dependence - {feat}')
    save_plot(f'stage6_shap_dependence_{feat}')
    plt.show()


## Step 3 - Applicant Report Function
Create `generate_applicant_report(...)` with exact required output keys and confidence logic.

In [ ]:
feature_to_english = {col: col.replace('_', ' ').capitalize() for col in X_test_eng.columns}
feature_to_english.update({
    'utility_payment_ratio': 'Utility bill payment consistency',
    'income_regularity_index': 'Income stability over 6 months',
    'loan_to_income_ratio': 'Loan amount relative to repayment capacity',
    'digital_footprint_density': 'Digital activity data completeness',
    'upi_consistency_score': 'UPI usage consistency',
    'financial_discipline_score': 'Financial discipline from psychometric answers',
    'future_planning_score': 'Future planning orientation',
    'risk_appetite_score': 'Risk aversion psychometric signal',
})

cluster_profiles = {int(k): v for k, v in cluster_names.items()}

def risk_tier_from_probability(p):
    if p < 0.30:
        return 'Low'
    if p <= 0.60:
        return 'Medium'
    return 'High'

def generate_applicant_report(applicant_index, X_test_eng, shap_values, y_test, cluster_model, cluster_profiles):
    row = X_test_eng.iloc[[applicant_index]]
    row_vec = shap_values[applicant_index]
    prob = float(best_model.predict_proba(row)[:, 1][0])
    sorted_idx = np.argsort(row_vec)
    pos_idx = sorted_idx[:3]
    neg_idx = sorted_idx[-3:][::-1]
    top_pos = [feature_to_english.get(X_test_eng.columns[i], X_test_eng.columns[i]) for i in pos_idx]
    top_neg = [feature_to_english.get(X_test_eng.columns[i], X_test_eng.columns[i]) for i in neg_idx]
    missing_flags = int(row['upi_data_missing'].iloc[0] + row['rent_data_missing'].iloc[0] + row['ecomm_data_missing'].iloc[0])
    density = float(row['digital_footprint_density'].iloc[0])
    if density >= 0.8 and row['upi_data_missing'].iloc[0] == 0 and row['ecomm_data_missing'].iloc[0] == 0:
        confidence = 'High'
    elif density < 0.4 or missing_flags >= 2:
        confidence = 'Low'
    else:
        confidence = 'Medium'
    cluster_id = int(cluster_model.predict(row[stage5['clustering_features']])[0])
    return {
        'borrower_id': int(X_test_eng.index[applicant_index]),
        'default_probability': round(prob, 4),
        'risk_tier': risk_tier_from_probability(prob),
        'actual_outcome': int(y_test.iloc[applicant_index]),
        'top_3_positive_factors': top_pos,
        'top_3_negative_factors': top_neg,
        'data_confidence_flag': confidence,
        'cluster_id': cluster_id,
        'cluster_profile_name': cluster_profiles.get(cluster_id, 'Unknown Cluster Profile'),
    }

print('Feature mapping coverage:', len(feature_to_english), 'features mapped.')


## Step 4 - Waterfall Plots
Generate SHAP waterfall plots for one defaulter and one non-defaulter.

In [ ]:
defaulter_idx = int(np.where(y_test.values == 1)[0][0])
non_defaulter_idx = int(np.where(y_test.values == 0)[0][0])

base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = float(np.array(base_val).reshape(-1)[-1])

for idx, label in [(defaulter_idx, 'defaulter_example'), (non_defaulter_idx, 'non_defaulter_example')]:
    exp = shap.Explanation(values=shap_values[idx], base_values=base_val, data=X_test_eng.iloc[idx], feature_names=X_test_eng.columns)
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(exp, max_display=12, show=False)
    plt.title(f'Stage 6 Waterfall Plot - {label}')
    save_plot(f'stage6_waterfall_{label}')
    plt.show()
print('Chosen indices:', {'defaulter': defaulter_idx, 'non_defaulter': non_defaulter_idx})


## Step 5 - Natural Language Summary Function
Create `generate_nl_summary(report_dict)` and demonstrate it on the same two applicants.

In [ ]:
def generate_nl_summary(report_dict):
    risk = report_dict['risk_tier']
    borrower_type = borrower_type_test.loc[report_dict['borrower_id']] if report_dict['borrower_id'] in borrower_type_test.index else 'unknown'
    pos = report_dict['top_3_positive_factors'][:2]
    neg = report_dict['top_3_negative_factors'][:2]
    confidence = report_dict['data_confidence_flag']
    if risk == 'Low':
        recommendation = 'Recommended for approval'
    elif risk == 'Medium':
        recommendation = 'Recommended for manual review'
    else:
        recommendation = 'Recommended for rejection'
    return (
        f"This {borrower_type} borrower is assessed as {risk} risk. "
        f"The strongest protective factors are {pos[0]} and {pos[1]}. "
        f"The strongest risk-elevating factors are {neg[0]} and {neg[1]}. "
        f"Data confidence is {confidence}, which indicates the reliability of observed digital and behavioral inputs. "
        f"{recommendation}."
    )

report_def = generate_applicant_report(defaulter_idx, X_test_eng, shap_values, y_test, cluster_model, cluster_profiles)
report_non = generate_applicant_report(non_defaulter_idx, X_test_eng, shap_values, y_test, cluster_model, cluster_profiles)
print('Defaulter report:', report_def)
print('Defaulter NL summary:', generate_nl_summary(report_def))
print('\nNon-defaulter report:', report_non)
print('Non-defaulter NL summary:', generate_nl_summary(report_non))


## Save Stage 6 Outputs

In [ ]:
stage6_output = {
    **stage5,
    'shap_values': shap_values,
    'top3_features': top3_features,
    'defaulter_report': report_def,
    'non_defaulter_report': report_non,
}
out_path = PROCESSED_DIR / 'stage6_outputs.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(stage6_output, f)
print('Saved:', out_path)


## Stage 6 Summary

In [ ]:
print('=' * 70)
print('STAGE 6 SUMMARY')
print('- Computed SHAP values and generated global/local explainability outputs.')
print(f"- SHAP matrix shape: {np.array(shap_values).shape}")
print(f"- Top global SHAP features: {top3_features}")
print(f"- Example reports generated for borrower IDs: {report_def['borrower_id']}, {report_non['borrower_id']}")
print(f"- Saved outputs to: {out_path}")
print('=' * 70)


In [ ]:
import numpy as np

feature_name_map = feature_to_english

shap_feature_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=X_test_eng.columns
).sort_values(ascending=False)

inference_s6 = {
    'explainer': explainer,
    'feature_name_map': feature_name_map,
    'shap_feature_importance': shap_feature_importance,
}
with open(PROCESSED_DIR / 'inference_artifacts_stage6.pkl', 'wb') as f:
    pickle.dump(inference_s6, f)
